<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/skipper-bradley-terry.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · Bradley–Terry skipper rankings

Rank skippers on **one strength scale that spans divisions**.

In fleet racing, A-division and B-division skippers sail *separate fleets* — an
A skipper never starts against a B skipper. Any per-division rating (like the
Elo in `skipper-elo.ipynb`) therefore can't say whether the best A skipper is
stronger than the best B skipper; the production engine bolts on a
cross-division calibration offset to compensate.

Bradley–Terry fixes this structurally. It models only pairwise outcomes —
*"i finished ahead of j in a race they both sailed"* — and fits one strength
per skipper so that

$$P(i \text{ beats } j) = \frac{p_i}{p_i + p_j}$$

Strength propagates through the whole **comparison graph**: the divisions are
bridged by every sailor who sailed both A and B at some point in the year, so
everyone connected to the main component lands on a single coherent scale.
Unlike Elo, the fit is order-invariant — a season-aggregate strength, not a
form-over-time trajectory (the Elo notebook keeps that story).

## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"

## Filters

Same pool as `skipper-elo.ipynb`: skippers only, one participant category,
non-counting regatta types and non-fleet boat classes excluded. No grades or
K-factors here — Bradley–Terry weighs every comparison equally (grade
weighting is a natural extension; see the closing note).

In [ ]:
# Types that never count toward ratings
EXCLUDE_TYPES = {
    "Scrimmage Regatta",
    "NOT ALLOWED  Promotional Regatta",
    "NOT ALLOWED  In-Conference Regatta",
}

# Keelboat / offshore / singlehanded / match-race classes excluded from fleet ratings
EXCLUDED_BOATS = (
    "J-70",
    "J-22",
    "Catalina 37",
    "Navy 44",
    "Colgate 26",
    "Sonar",
    "Tartan 10",
    "Melges 24",
    "Flying Scot",
    "Rhodes",
    "Laser",
    "Radial",
    "Match Race 3-4",
)

PROVISIONAL_RACES = 20  # below this many races, a rating is flagged low-confidence
ALPHA = 0.1  # smoothing: pseudo-wins added each way on every observed pair

# The full academic year.
SEASONS = ["f25", "s26"]
PARTICIPANT = "coed"  # or "women"

## Load the year and keep skipper-race rows

No chronological ordering needed — Bradley–Terry doesn't care when a
comparison happened, only that it did.

In [ ]:
import scraper

data = scraper.load(SEASONS, sailors=True, progress=True).fleet()
print(len(data.regattas), "fleet regattas |", len(data.sailor_races), "sailor-race rows")

In [ ]:
import pandas as pd

races = data.sailor_races_frame()
races = races[
    (races["boat_role"] == "skipper")
    & (races["sailor_slug"] != "")  # unresolved sailors would merge into one phantom skipper
    & (races["participant"] == PARTICIPANT)
    & (~races["regatta_type"].isin(EXCLUDE_TYPES))
    & (~races["boat"].isin(EXCLUDED_BOATS))
].copy()
print(len(races), "skipper-race rows after filtering")

## Every race becomes pairwise outcomes

Within one race (same regatta, division, race number), each pair of boats is
one comparison, won by the lower scored place. Exact ties in scored points
contribute nothing. An 18-boat race yields 153 comparisons — the counts add
up fast, so we aggregate to per-pair totals as we go.

In [ ]:
from collections import Counter, defaultdict

wins = Counter()  # (winner_slug, loser_slug) -> comparisons won
races_count = Counter()  # boat-races per skipper
divisions_sailed = defaultdict(Counter)  # skipper -> {division: races}

race_key = ["season", "regatta_slug", "division", "race_num"]
for (_, _, division, _), g in races.groupby(race_key, sort=False):
    boats = sorted(zip(g["place"], g["sailor_slug"]))
    for i, (pi, si) in enumerate(boats):
        races_count[si] += 1
        divisions_sailed[si][division] += 1
        for pj, sj in boats[i + 1 :]:
            if pi < pj:  # equal places (ties) contribute nothing
                wins[(si, sj)] += 1

print(f"{sum(wins.values()):,} pairwise outcomes across {len(wins):,} directed pairs")

## Connectivity: how A and B get onto one scale

Bradley–Terry can only compare skippers who are connected in the comparison
graph — directly (raced each other) or through chains of shared opponents.
The A/B divide would split the graph in two **if nobody ever switched
divisions** — but sailors do sail both across a year, and each one of them is
a bridge that lets strength propagate between the pools.

We fit only the largest connected component (isolated cliques have no basis
for comparison with it) and report who the bridges are.

In [ ]:
# Union-find over the (undirected) comparison graph.
parent: dict = {}


def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]  # path halving
        x = parent[x]
    return x


for a, b in wins:
    parent[find(a)] = find(b)

all_skippers = {s for pair in wins for s in pair}
components = Counter(find(s) for s in all_skippers)
main_root, main_size = components.most_common(1)[0]
skippers = sorted(s for s in all_skippers if find(s) == main_root)

bridges = [s for s in skippers if len(divisions_sailed[s]) > 1]
mix = Counter(
    "both" if len(divisions_sailed[s]) > 1 else next(iter(divisions_sailed[s])) for s in skippers
)

print(f"{len(all_skippers)} skippers, {len(components)} components")
print(f"Main component: {main_size} skippers ({main_size / len(all_skippers):.0%})")
print(f"Division mix in main component: {dict(sorted(mix.items()))}")
print(f"{len(bridges)} multi-division bridge sailors, e.g.: {bridges[:5]}")

## Fit: Hunter's MM algorithm

The maximum-likelihood strengths satisfy a fixed point that Hunter's (2004)
minorize–maximize update converges to monotonically:

$$p_i \leftarrow \frac{W_i}{\sum_{j} \dfrac{n_{ij}}{p_i + p_j}}$$

where $W_i$ is the total comparisons $i$ won and $n_{ij}$ the total
comparisons between $i$ and $j$. Two practical touches:

- **Smoothing**: a skipper who won (or lost) *every* comparison sends the MLE
  to infinity (or zero). Adding $\alpha$ pseudo-wins in each direction on
  every observed pair keeps all strengths finite — a tiny Bayesian shrink
  toward "everyone is average".
- **Normalization**: strengths are only identified up to a constant factor,
  so each iteration renormalizes to geometric mean 1. We report
  $400 \log_{10} p$ — the same logistic curve as Elo, so a 400-point gap
  again means 10:1 odds.

In [ ]:
import numpy as np

idx = {s: k for k, s in enumerate(skippers)}
m = len(skippers)

# Aggregate directed wins into unique undirected pairs (i < j by index).
pair_w: dict[tuple[int, int], list[float]] = {}
for (a, b), c in wins.items():
    if a not in idx or b not in idx:
        continue  # outside the main component
    i, j = idx[a], idx[b]
    key, flip = ((i, j), 0) if i < j else ((j, i), 1)
    entry = pair_w.setdefault(key, [ALPHA, ALPHA])  # smoothing both ways
    entry[flip] += c

ii = np.array([k[0] for k in pair_w])  # first skipper of each pair
jj = np.array([k[1] for k in pair_w])  # second skipper of each pair
w_ij = np.array([v[0] for v in pair_w.values()])  # wins by ii over jj
w_ji = np.array([v[1] for v in pair_w.values()])
n_pair = w_ij + w_ji

wins_total = np.zeros(m)
np.add.at(wins_total, ii, w_ij)
np.add.at(wins_total, jj, w_ji)

p = np.ones(m)
for iteration in range(1, 5001):
    shared = n_pair / (p[ii] + p[jj])
    denom = np.zeros(m)
    np.add.at(denom, ii, shared)
    np.add.at(denom, jj, shared)
    p_new = wins_total / denom
    p_new /= np.exp(np.mean(np.log(p_new)))  # geometric mean 1
    delta = np.max(np.abs(np.log(p_new) - np.log(p)))
    p = p_new
    if delta < 1e-8:
        break

print(f"converged after {iteration} iterations (max |dlog p| = {delta:.1e})")
rating = 400 * np.log10(p)  # Elo-like scale: +400 = 10:1 favourite

## Rankings — one scale, both divisions

Unlike Elo — where the K-factor caps how far a couple of races can move a
rating — the Bradley–Terry MLE will happily hand an extreme strength to a
skipper with two lucky races. So the headline table keeps only skippers with
at least `PROVISIONAL_RACES` races; the small-sample crowd is counted but not
ranked.

`A_share` is the fraction of a skipper's races sailed in A division: the
point of the exercise is that the top of the table can mix A regulars,
B regulars, and division-switchers on the same footing.

In [ ]:
names = races.drop_duplicates("sailor_slug").set_index("sailor_slug")[
    ["sailor_name", "school_slug"]
]

all_rated = pd.DataFrame(
    {
        "sailor_slug": skippers,
        "rating": np.round(rating, 1),
        "races": [races_count[s] for s in skippers],
        "A_share": [divisions_sailed[s]["A"] / sum(divisions_sailed[s].values()) for s in skippers],
    }
).merge(names, left_on="sailor_slug", right_index=True)

rankings = (
    all_rated[all_rated["races"] >= PROVISIONAL_RACES]
    .sort_values("rating", ascending=False)
    .reset_index(drop=True)
)
rankings.index = range(1, len(rankings) + 1)
print(f"{len(rankings)} ranked skippers ({len(all_rated) - len(rankings)} provisional excluded)")
rankings[["sailor_name", "school_slug", "rating", "races", "A_share"]].head(25)

## Chart the top 15

Bars colored by where the skipper mostly sailed — mixed colors at the top is
the cross-division scale doing its job.

In [ ]:
import matplotlib.pyplot as plt

top = rankings.head(15).iloc[::-1]


def _color(a_share):
    if a_share >= 0.8:
        return "tab:blue"  # A regular
    if a_share <= 0.2:
        return "tab:orange"  # B regular
    return "tab:green"  # sails both


plt.figure(figsize=(8, 6))
plt.barh(top["sailor_name"], top["rating"], color=[_color(a) for a in top["A_share"]])
for y, (r, a) in enumerate(zip(top["rating"], top["A_share"])):
    plt.text(r + 5, y, f"A {a:.0%}", va="center", fontsize=7)
plt.xlabel("Bradley-Terry rating (400 log10 p; +400 = 10:1 favourite)")
plt.title(f"{'+'.join(SEASONS)} ({PARTICIPANT}) - Bradley-Terry skipper strengths")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ("tab:blue", "tab:green", "tab:orange")]
plt.legend(handles, ["mostly A", "sails both", "mostly B"], loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

---
**Where this sits in the family:** Bradley–Terry is the pairwise special case
of Plackett–Luce (Harville's horse-racing model), which consumes whole
finishing orders instead of shredding them into pairs; Henery's variant swaps
the logistic for normal order statistics. Natural extensions here: weight
comparisons by regatta grade (nationals beat a Tuesday scrimmage), decay old
comparisons in time, or report per-skipper uncertainty from the Fisher
information. For the form-over-time story — who's rising, who's fading — see
`skipper-elo.ipynb`, which trades the single coherent scale for a trajectory.